# Mocks - unittest.mock

[Documentation](https://docs.python.org/3/library/unittest.mock.html)

`unittest.mock` is a library for testing in Python. It allows you to replace parts of your system under test with mock objects and make assertions about how they have been used.

`Mock` and `MagicMock` objects create all attributes and methods as you access them and store details of how they have been used. You can configure them, to specify return values or limit what attributes are available, and then make assertions about how they have been used:

In [4]:
from unittest.mock import MagicMock

class ProductionClass:
    def method(self, *args, **kwargs):
        return 'real stuff'
    
    def other_method(self):
        return 'not mocked'

thing = ProductionClass()
thing.method = MagicMock(return_value=3)
thing.method(3, 4, 5, key='value')

3

In [5]:
thing.other_method()

'not mocked'

In [8]:
thing.method.assert_called_with(3, 4, 5, key='value')

`side_effect` allows you to perform side effects, including raising an exception when a mock is called:

In [3]:
from unittest.mock import Mock

mock = Mock(side_effect=KeyError('foo'))
mock()

KeyError: 'foo'

A `side_effect` can also be a callable; when calling the mock, that callable will be called:

In [4]:
values = {'a': 1, 'b': 2, 'c': 3}

# def get_value(key):
#     return values[key]
# mock.side_effect = get_value

mock.side_effect = lambda x: values[x]
mock('a'), mock('b'), mock('c')

(1, 2, 3)

A `side_effect` can be an iterable of return values:

In [5]:
mock.side_effect = [5, 4, 3, 2, 1]
mock(), mock(), mock()

(5, 4, 3)

The difference between `Mock` and `MagicMock` is that `MagicMock` implements some of the most common magic / dunder methods, while `Mock` doesn't:

In [18]:
from unittest.mock import MagicMock

mock = MagicMock()
mock.__str__.return_value = 'foo'
str(mock)

'foo'

In [17]:
mock.__str__.assert_called_once()

Magic methods can also be used on `Mock` objects, but they have to be explicitly added:

In [19]:
from unittest.mock import Mock

mock = Mock()
mock.__str__ = Mock(return_value='foo')
str(mock)

'foo'

In [20]:
mock.__str__.assert_called_once()

The `patch()` decorator / context manager makes it easy to mock classes or objects in a module under test. The object you specify will be replaced with a mock (or other object) during the test and restored when the test ends:

In [22]:
from unittest.mock import patch

def func1(arg1):
    print('Doing some operations that need to be mocked', arg1)

def func2(arg2):
    value = arg2 + 1
    func1(value)
    return value

@patch('__main__.func1')
def test(func1_mock):
    result = func2(1)
    assert result == 2
    func1_mock.assert_called_once_with(2)
    
test()